# 01 — Análise exploratória da camada gold

Exploração e relatório apenas: a lógica de produção está em `src/`.
Pré-requisito: `make ingest prepare` (gera `data/gold/enem_gold.parquet`).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from constants.artifacts import GOLD_FILE
from constants.labels import INCOME_GROUP_ORDER
from visualization.theme import apply_theme

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
plt.rcParams["figure.dpi"] = 120
apply_theme()

In [ ]:
gold = pl.read_parquet(ROOT / "data" / "gold" / GOLD_FILE)
print(gold.shape)
gold.select("year", "mean_score").describe()

In [ ]:
funnel = pl.read_csv(ROOT / "reports" / "tables" / "participation_funnel.csv")
funnel

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for group in INCOME_GROUP_ORDER:
    scores = gold.filter(pl.col("income_group") == group)["mean_score"].to_numpy()
    if len(scores) > 1:
        sns.kdeplot(x=scores, label=group, ax=ax)
ax.set_title("Distribuição da nota média por faixa de renda")
ax.set_xlabel("Nota média (pontos)")
ax.set_ylabel("Densidade")
ax.legend(title="Renda familiar")
plt.tight_layout()